# 06 - Calcolatore: mi sono veramente pagato la pensione?

Questo notebook implementa un calcolatore didattico.

L'obiettivo e' confrontare:

- il tasso di sostituzione teoricamente sostenibile con i contributi accumulati;
- il tasso di sostituzione effettivo o ipotizzato;
- la quota della pensione che risulta non coperta dal montante contributivo simulato.

Le ipotesi sono modificabili. I risultati non sono una stima ufficiale.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CODE_DIR = REPO_ROOT / 'code'
METADATA_DIR = REPO_ROOT / 'metadata'
DATA_FINAL_DIR = REPO_ROOT / 'data' / 'final'

if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

from calcolatore_pensione_pagata import esegui_scenario, SCENARIO_BASE

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

## Scenari disponibili

Gli scenari sono in `metadata/scenari_calcolatore_pensione_pagata.csv`. Ogni riga contiene ipotesi modificabili su carriera, aliquote, capitalizzazione, pensionamento e grandezze aggregate.

In [ ]:
scenari_path = METADATA_DIR / 'scenari_calcolatore_pensione_pagata.csv'
scenari = pd.read_csv(scenari_path)
display(scenari)

## Scegli scenario

Modifica `SCENARIO_ID` per usare un altro scenario. Puoi anche sovrascrivere singole ipotesi nel dizionario `scenario`.

In [ ]:
SCENARIO_ID = 'scenario_video_didattico'

riga = scenari.loc[scenari['scenario_id'].eq(SCENARIO_ID)].iloc[0].to_dict()
scenario = dict(SCENARIO_BASE)

for chiave in scenario:
    if chiave in riga and pd.notna(riga[chiave]):
        scenario[chiave] = riga[chiave]

# Esempi di modifica manuale:
# scenario['tasso_sostituzione_effettivo'] = 0.79
# scenario['speranza_vita_residua'] = 20
# scenario['aliquota_contributiva'] = 0.33

scenario

## Esecuzione del calcolatore

La carriera usa un salario indicizzato, i contributi annui e un montante contributivo figurativo. La pensione teorica e' il montante diviso per la speranza di vita residua.

In [ ]:
carriera, risultati = esegui_scenario(scenario)
display(carriera.head())
display(carriera.tail())
display(risultati.T.rename(columns={0: 'valore'}))

## Lettura del risultato

- `tasso_sostituzione_teorico`: quanto si potrebbe pagare in base al montante simulato.
- `tasso_sostituzione_effettivo`: tasso osservato o ipotizzato.
- `quota_pensione_non_coperta`: parte della pensione effettiva che eccede la pensione teorica del modello.
- `quota_non_coperta_per_occupato`: valore medio per occupato se la quota viene applicata alla spesa aggregata.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(carriera['anno'], carriera['salario'], marker='o', label='Salario')
ax.plot(carriera['anno'], carriera['contributi'], marker='o', label='Contributi')
ax.set_title('Carriera simulata')
ax.set_xlabel('Anno')
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(carriera['anno'], carriera['montante_contributivo'], marker='o')
ax.set_title('Montante contributivo simulato')
ax.set_xlabel('Anno')
ax.set_ylabel('Indice')
plt.tight_layout()
plt.show()

## Sensitivity analysis

Qui si cambia la speranza di vita residua. A parita' di montante, una vita residua piu lunga riduce la pensione annua teoricamente sostenibile.

In [ ]:
righe = []
for speranza in [17, 20, 22, 25.5]:
    scenario_sens = dict(scenario)
    scenario_sens['speranza_vita_residua'] = speranza
    _, risultati_sens = esegui_scenario(scenario_sens)
    riga = risultati_sens.iloc[0].to_dict()
    riga['scenario_speranza_vita'] = speranza
    righe.append(riga)

sensibilita = pd.DataFrame(righe)
display(sensibilita[['scenario_speranza_vita', 'tasso_sostituzione_teorico', 'quota_pensione_non_coperta', 'quota_non_coperta_per_occupato']])

## Salvare output

Questa cella salva carriera e risultati in `data/final/`.

In [ ]:
DATA_FINAL_DIR.mkdir(parents=True, exist_ok=True)
carriera.to_csv(DATA_FINAL_DIR / 'calcolatore_pensione_pagata_carriera.csv', index=False)
risultati.to_csv(DATA_FINAL_DIR / 'calcolatore_pensione_pagata_base.csv', index=False)
print('Output salvati in data/final/.')